In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import pickle

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

In [2]:
pizza_df = pd.read_csv(r"E:/GUVI Projects/Dominos/Pizza_Sale.csv")
ingred_df = pd.read_csv(r"E:/GUVI Projects/Dominos/Pizza_Ingredients.csv")

In [3]:
pizza_df.dropna(inplace=True)
pizza_df['order_date'] = pizza_df['order_date'].apply(
    lambda x: pd.to_datetime(x, format='%d-%m-%Y', errors='coerce') 
    if pd.to_datetime(x, format='%d-%m-%Y', errors='coerce') is not pd.NaT 
    else pd.to_datetime(x, format='%d/%m/%Y', errors='coerce')
)
pizza_df = pizza_df[['order_date', 'pizza_name', 'quantity']]

In [4]:
sales_summary = pizza_df.groupby(['order_date', 'pizza_name'])['quantity'].sum().reset_index()
sales_pivot = sales_summary.pivot(index='order_date', columns='pizza_name', values='quantity').fillna(0)

In [5]:
arima_models = {}
for pizza in sales_pivot.columns:
    try:
        model = ARIMA(sales_pivot[pizza], order=(1, 1, 0))
        model_fit = model.fit()
        arima_models[pizza] = model_fit
        
        # Save each model individually for future use
        filename = f"arima_model_{pizza.replace(' ', '_')}.pkl"
        with open(filename, 'wb') as file:
            pickle.dump(model_fit, file)
    except Exception as e:
        print(f'ARIMA model for {pizza} failed: {e}')

In [6]:
forecast_days = 7
predictions = {}
for pizza, model_fit in arima_models.items():
    try:
        predictions[pizza] = model_fit.predict(start=len(sales_pivot), end=len(sales_pivot) + forecast_days - 1)
    except Exception as e:
        print(f'Forecasting failed for {pizza}: {e}')
predictions_df = pd.DataFrame(predictions)
predictions_df.index = pd.date_range(start=sales_pivot.index[-1] + pd.Timedelta(days=1), periods=forecast_days, freq='D')
print("Forecasted Sales for Next 7 Days:")
print(predictions_df)

Forecasted Sales for Next 7 Days:
            The Barbecue Chicken Pizza  The Big Meat Pizza  \
2016-01-01                    7.949231            4.420315   
2016-01-02                    9.000644            5.252117   
2016-01-03                    8.461593            4.814122   
2016-01-04                    8.737960            5.044753   
2016-01-05                    8.596269            4.923311   
2016-01-06                    8.668913            4.987258   
2016-01-07                    8.631669            4.953586   

            The Brie Carre Pizza  The Calabrese Pizza  \
2016-01-01              0.868806             1.368239   
2016-01-02              1.508606             2.255787   
2016-01-03              1.146737             1.773032   
2016-01-04              1.351409             2.035612   
2016-01-05              1.235647             1.892789   
2016-01-06              1.301122             1.970474   
2016-01-07              1.264089             1.928219   

            

In [7]:
ingred_df.rename(columns={'Items_Qty_In_Grams': 'items_qty'}, inplace=True)

ingredient_quantities = {}
for pizza in predictions_df.columns:
    predicted_qty = predictions_df[pizza].sum()
    pizza_ing = ingred_df[ingred_df['pizza_name'] == pizza]
    for _, row in pizza_ing.iterrows():
        ingredient = row['pizza_ingredients']
        ing_qty = row['items_qty']
        ingredient_quantities[ingredient] = ingredient_quantities.get(ingredient, 0) + (predicted_qty * ing_qty)

purchase_order_df = pd.DataFrame.from_dict(ingredient_quantities, orient='index', columns=['quantity'])
purchase_order_df['unit'] = 'grams'

print("\nFinal Purchase Order:")
print("-----------------------")
print(purchase_order_df.to_string())


Final Purchase Order:
-----------------------
                                quantity   unit
Barbecued Chicken            5404.165210  grams
Red Peppers                 11341.551998  grams
Green Peppers                8030.393870  grams
Tomatoes                    34984.718341  grams
Red Onions                  54797.556512  grams
Barbecue Sauce               1801.388403  grams
Bacon                       19992.004764  grams
Pepperoni                   24192.916429  grams
Italian Sausage               343.954622  grams
Chorizo Sausage              1719.773109  grams
Brie Carre Cheese             260.292444  grams
Prosciutto                    260.292444  grams
Caramelized Onions                   NaN  grams
Pears                          86.764148  grams
Thyme                          43.382074  grams
Garlic                      17939.075392  grams
?duja Salami                 1586.898271  grams
Pancetta                     2380.347406  grams
Friggitello Peppers           396.724568 